# Python编程基础 · 工单1.5《工业互联网数据采集系统》 — 教师参考版
第11周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [ ]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


## 参考实现 `w11`（含依赖导入）

In [ ]:
import json
import random
import re
import sys
import numpy as np  # 通用科学计算


def w11():
    import time
    import json
    from pymodbus.client import ModbusTcpClient as Client
    import paho.mqtt.client as mqtt

    c = Client(host="127.0.0.1", port=5020)
    c.connect()
    mq = mqtt.Client()
    mq.connect("127.0.0.1", 1883)
    mq.loop_start()
    names = ["temp_c", "press_mpa", "vib_mm_s", "rpm", "status"]
    for i in range(5):
        rr = c.read_holding_registers(address=0, count=5)
        vals = rr.registers
        data = {n: (v / 10 if n != "status" else v)
                for n, v in zip(names, vals)}
        payload = json.dumps({"device_id": "DEV-1",
                              "ts": time.strftime("%H:%M:%S"),
                              "metrics": data}, ensure_ascii=False)
        mq.publish("device/1/status", payload)
        print("上报:", payload)
        time.sleep(2)
    mq.loop_stop()
    c.close()


## 运行环境(如需)

In [ ]:
# 启动虚拟产线(离线模拟器, 5轮采集约15秒)
import subprocess, sys, time, atexit
p1 = subprocess.Popen([sys.executable, "虚拟产线/mini_modbus_server.py", "5020", "1"])
p2 = subprocess.Popen([sys.executable, "虚拟产线/mini_mqtt_broker.py", "1883"])
atexit.register(lambda: (p1.terminate(), p2.terminate()))
time.sleep(1.2)
print("模拟器就绪: PLC@5020 / MQTT@1883")

## 执行与验证

In [ ]:
w11()